# Python-Übungen für den 15.10.2026

*Teil der Lehrveranstaltung «Die Sprache der Politik», HS 2026*

In diesem zweiten Notebook wird vor allem die Bibliothek **spaCy** eingeführt, sowie noch einige zusätzliche Elemente der Sprache Python.

Wie im ersten Notebook gibt es Code-Zellen, die ausgeführt werden sollen, und andere, die für die Aufgaben / Übungen vorgesehen sind.

## 1 spaCy

Neben den Modulen, die Teil jeder Python-Installation sind, wie etwa `os`, gibt es auch sogenannte Bibliotheken (*libraries*), die man zusätzlich installieren kann. In der Colab-Umgebung sind einige dieser Bibliotheken, die sehr bekannt sind und in «data science»-Projekten oft benutzt werden, schon installiert.

Dazu gehören auch Bibliotheken, die von Linguist:innen entwickelt worden sind, um Sprachen zu analysieren. Oft wird in diesem Kontext von *natural language processing* oder kurz NLP gesprochen («natürliche Sprache» im Gegensatz etwa zu Programmiersprachen).

Zwei sehr bekannte Bibliotheken für NLP sind **NLTK** (Natural Language Toolkit), die auch im Blogpost von Ted Underwood kurz erwähnt wurde, und **spaCy**, die wir jetzt gemeinsam anschauen werden.

Als erste Aufgabe schau Dir kurz die Webseite von [spaCy](https://spacy.io/) an.

Da spaCy in Google Colab schon installiert ist, kann die Bibliothek einfach importiert werden:

In [1]:
import spacy

### 1.1 Sprachmodelle herunterladen

Um das Potenzial von spaCy richtig ausschöpfen zu können, muss ein Sprachmodell heruntergeladen werden. Die vortrainierte Sprachmodelle, die angeboten werden, sind zwar nicht so gross wie die «grossen» (eigentlich müsste man sagen: riesigen) Sprachmodelle der sog. «KI», d.h. die LLMs; dennoch sind sie doch zu gross, um direkt mit der Bibliothek geliefert zu werden.

Diese vortrainierte Modelle gibt es für viele Sprachen. Für uns vor allem relevant: es gibt deutsche und französische Sprachmodelle. Sie kommen in zwei Varianten: ein kleineres Modell, das schneller arbeitet, aber weniger genau; ein grösseres Modell, das bessere Ergebnisse liefert, aber langsamer ist. In beiden Fällen bleiben sie aber deutlich kleiner als Modelle von LLMs und können auch ohne GPU laufen. Man kann sie also in Google Colab auch gratis laufen lassen.

*Für die Übungen in diesem Notebook nutzen wir ein kleines Modell. Für die Gruppenarbeit kann es sich je nach dem lohnen, mit einem grösseren Modell zu arbeiten.*

Standard ist es, diese Sprachmodelle über die Kommandozeile herunterzuladen. So wird das auch auf der [entsprechenden Seite der spaCy-Dokumentation](https://spacy.io/usage/models) vorgeschlagen:

`python -m spacy download de_core_news_sm`

In Google-Colab ist eine Kommandozeile über den Knopf «Terminal» (unten links im Fenster) verfügbar. In Jupyter Notebooks (und somit auch in Google Colab) ist es aber auch möglich, in einer Codezelle anstatt Python ein Kommando auszuführen, indem man eine Zelle mit einem Ausrufezeichen beginnt und sofort darauf ein Kommando eingibt.

Das machen wir jetzt auch in der folgenden Zelle:

In [2]:
!python -m spacy download de_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 33.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('de_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


SpaCy gibt an, dass möglicherweise die Sitzung neu gestartet werden muss:

> If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.

In meiner Erfahrung war das nicht nötig. Sollte die nächste Zelle aber nicht funktionieren, solltest Du auf `Laufzeit ▶ Sitzung neu starten` zugreifen.

### 1.2 Das Sprachmodell laden

Sobald man ein Sprachmodell heruntergeladen hat, ist die Nutzung von spaCy leicht. Dafür braucht es eigentlich nur eine Zeile, die das Sprachmodell lädt:

```python
nlp = spacy.load("de_core_news_sm")
```

Danach kann die dadurch geladene neue Funktion `nlp` aufgerufen werden, mit einer Zeichenkette als Argument. Was sie zurückgibt ist ein `Doc`-Objekt mit vielen nützlichen Methoden.

Probieren wir es aus:

In [3]:
nlp = spacy.load("de_core_news_sm")

# Der Satz kommt aus dem Amtlichen Bulletin!
text1 = "Der Berg hat eine Maus geboren! "
doc1 = nlp(text1)

# Auch das kommt aus dem Amtlichen Bulletin (wohl weniger überraschend)
text2 = "Der Artikel 48 des Bundesgesetzes vom 1. Oktober 1925 über das Zollwesen, bzw. Artikel 111 der bezüglichen Vollziehungsverordnung sehen im Abfertigungsverfahren eine ganze Anzahl Vereinfachungen und Erleichterungen vor. Der Artikel 14 des gleichen Gesetzes, bzw. Artikel 11 der Vollziehungsverordnung sagt uns, was und welche Waren-Sendungen und Warenmengen zollfrei seien."
doc2 = nlp(text2)

In den nächsten Unterkapiteln und Codezellen werden wir einige der Möglichkeiten, die dieses Objekt anbietet. Wir werden dabei oft mit `text1`, `doc1`, `text2` und `doc2` arbeiten.

### 1.3 Tokens: In Wörtern zergliedern

Bisher haben wir einfach die Methode `.split()` verwendet, um einen Text in Wörtern zu zergliedern:

In [4]:
text1.split(" ")

['Der', 'Berg', 'hat', 'eine', 'Maus', 'geboren!', '']

Dabei ist uns schon aufgefallen, dass Satzzeichen (hier der Ausrufezeichen) Teil der «Wörter» sind, weil wir die Zeichenkette einfach entlang der Leerzeichen zergliedert wird. In diesem Beispiel sehen wir auch, dass wir als letztes «Wort» eine leere Zeichenkette zurückbekommen, weil `text1` mit einem Leerzeichen endet.

Es gibt zwar Möglichkeiten, mit Standardfunktionen von Python, wie z. B. `.strip()`, damit umzugehen. Die spezialisierte Bibliothek spaCy bietet aber da Möglichkeiten an, die durch einfachen Code verwendet werden können, und die das Problem besser lösen.

Das `Doc`-Objekt, das von `nlp()` zurückgegeben wird, kann man nutzen, als wäre es eine Liste. Jedes Element dieser Liste ist ein sogenanntes `Token`. Tokens sind kleine [«logisch zusammengehörigen Einheiten»](https://de.wikipedia.org/wiki/Lexikalische_Analyse). Insbesondere Wörter gehören zu den Tokens.

Schauen wir uns an, was das erste Token in `doc1` ist:

In [5]:
doc1[0]

Der

Jetzt nutzen wir eine `for`-Schleife, um alle Tokens in `doc1` anzuzeigen:

In [6]:
for token in doc1:
  print(token.text)

Der
Berg
hat
eine
Maus
geboren
!


Wie wir sehen, sind Tokens nicht nur Wörter: auch das Satzzeichen `!` wird separat aufgelistet. Das macht auch Sinn: SpaCy will eine sprachliche Analyse durchführen, und Satzzeichen sind auch wichtige Elemente der schriftlichen Sprache.

Die Tokens von spaCy sind nicht einfach Zeichenketten, sondern Objekte, die verschiedene Attribute haben. Dazu zählt das Attribut `.text`, das wir gerade benutzt haben. Es beinhaltet die Zeichenkette (string) des Tokens.

<br/>

---

📖 Attribute sind ein Teil von Python, das noch nicht eingeführt worden ist. Es handelt sich einfach um Werte, die an einem Objekt angegliedert sind. Ähnlich wie Methoden wird auf sie so zugegriffen: `object.attribute`. Im Unterschied zu Methoden stehen nach dem Wort keine runde Klammern, da sie keine Funktionen sind, sondern einfach Werte.<br/>

---

Zu den Attributen von Tokens gehört etwa auch `.i`, das die Stelle des Tokens innerhalb des `Doc`-Objektes angibt.

Ein Beispiel dazu:

In [ ]:
maus_index = None
for token in doc1:
  if token.text == "Maus":
    maus_index = token.i
    print(f"The index of 'Maus' in `doc1` is: {maus_index}")

Die `Token`-Objekte haben aber auch andere Attribute. Die ganze Liste ist in der [Dokumentation](https://spacy.io/api/token#attributes) zu finden.

Sehen wir uns jetzt die Attribute an, die für uns besonders nützlich sind.

#### 1.3.1 Stoppwörter und Satzzeichen ignorieren

Wir haben es im Kurs schon angesprochen: in einem Text sind viele kleine Wörter, die Computer-Linguist:innen als [«Stoppwörter»](https://de.wikipedia.org/wiki/Stoppwort) bezeichnen. Das sind Wörter, die sprachlich sehr wichtig sind, sehr häufig auftreten aber wenig über den Inhalt einer Aussage informieren. Das sind z.B. Artikel (der, die, das, ein, …), Pronomen (er, sie, es) oder häufige Verben (sein, haben).

Je nach Fragestellung können diese Wörter sehr relevant sein (eine Berücksichtigung der Syntax ist ohne diese Wörter unmöglich) oder die Analyse eher stören (wenn wir etwa wissen wollen, welches Wort am meisten in einem Text vorkommt, und uns «er» oder «der» nicht interessieren).

Das Attribut `.is_stop` der `Token`-Objekte gibt an, ob es sich beim Token um ein Stoppwort handelt:

In [7]:
for token in doc1:
  if token.is_stop:
    print(token.text)

Der
hat
eine


Nehmen wir an, dass wir eine Liste der Wörter in `doc1` erstellen möchten, ohne Stoppwörtern. Da nicht nur eigentliche Wörter zu den Tokens zählen, sondern auch Satzzeichen, müssen wir auch wissen, ob ein Token ein Satzzeichen ist. Da hilft das Attribut `.is_punct`:

In [8]:
for token in doc1:
  print(f"Das Token: '{token.text}' ist ein Satzzeichen: {token.is_punct}")

Das Token: 'Der' ist ein Satzzeichen: False
Das Token: 'Berg' ist ein Satzzeichen: False
Das Token: 'hat' ist ein Satzzeichen: False
Das Token: 'eine' ist ein Satzzeichen: False
Das Token: 'Maus' ist ein Satzzeichen: False
Das Token: 'geboren' ist ein Satzzeichen: False
Das Token: '!' ist ein Satzzeichen: True


Mit `.is_stop` und `is_punct` lässt sich jetzt die Liste der Wörter in `doc1`, ohne Stoppwörtern und ohne Satzzeichen, erstellen. Ergänze den folgenden Codeblock:

In [ ]:
# Aufgabe 1.4
# Ersetze im Code die ____
# Das Ziel ist, in `filtered_words` eine Liste von Zeichenketten zu haben, die

filtered_words = []

for ______ in ____:
  if not _____.is_stop and not _____.is_punct:
    filtered_words.append(_____.text)

filtered_words

Hier die Antwort:

<details>
<summary>Antwort aufklappen</summary>

```python
filtered_words = []

for token in doc1:
  if not token.is_stop and not token.is_punct:
    filtered_words.append(token.text)

filtered_words
```

</details>

#### 1.3.2 Lemmatisierung

Stellen wir uns vor, dass uns das Verb «gebären» im Satz «Der Berg hat eine Maus geboren!» besonders interessiert. Wir möchten jetzt untersuchen, in welchen Parlamentsdebatten das Verb «gebären» vorkam und in welchem Kontext. Das Problem ist aber, das wir alle Formen des Verbs suchen wollen, und nicht nur das Wort «gebären» oder die Form «geboren». Anstatt selber eine Liste von Formen zu erstellen, können wir das Sprachmodell von spaCy verwenden.

Jedes Token hat nämlich das Attribut `.lemma_`, das das [Lemma](https://de.wikipedia.org/wiki/Lemma_(Lexikographie)), das heisst die Grundform eines Wortes angibt.

(Aufpassen: es handelt sich um das Attribut `.lemma_`, mit Unterstrich am Ende, und nicht `.lemma`. Das erste ist eine Zeichenkette und fast immer das, was uns interessiert. Das zweite ist eine Zahl, die von spaCy benutzt wird aber uns nicht interessiert.)

Schauen wir uns an, welche Lemmata (ja, so heisst der Plural von Lemma) die Tokens in `doc1` haben:

In [9]:
for token in doc1:
  print(f"{token.text} -> {token.lemma_}")

Der -> der
Berg -> Berg
hat -> haben
eine -> ein
Maus -> Maus
geboren -> gebären
! -> --


Ja nach Analyse oder Fragestellung kann es Sinn machen, eher die Lemmata anzuschauen. Durch die Lemmatisierung werden auch Singular- und Pluralformen vereinheitlicht. So wäre es zum Beispiel möglich, alle Protokolle zu finden, in denen «Berg», «Berge» und «Bergen» vorkommen (bzw. zählen, wie oft das Wort «Berg» in seinen verschiedenen Deklinierungen vorkommt).

Natürlich lassen sich verschiedene Analyse kombinieren. Vielleicht kommt man erst zu interessanten Einsichten, wenn man nicht nur zählt, wie oft das Wort «Berg/Berge/Bergen» vorkommt, sondern auch, wie oft und wann jede einzelne Form vorkommt (also ohne Lemmatisierung), und vielleicht auch, wieviele Wörtern vorkommen, die mit «Berg» anfangen, wie «Bergbahnen».

Es sei angemerkt, dass die technische Möglichkeiten hier das Denken nicht ersetzen. Etwa: andere Wörter müssten in diesem Beispiel vielleicht auch mitberücksichtigt werden, wie «Gebirge»…

#### 1.3.3 Part-of-speech tagging

`Token`-Objekte haben noch viele andere nützliche Attribute und Funktionen. Es sei hier zum Schluss einfach noch auf die Attribute `.tag_` und `.pos_` hingewiesen.

Schau Dir zuerst [die Erklärungen in der Dokumentation von spaCy an](https://spacy.io/usage/linguistic-features#pos-tagging).

Dort heisst es unter anderem:

> This is where the trained pipeline and its statistical models come in, which enable spaCy to make predictions of which tag or label most likely applies in this context. A trained component includes binary data that is produced by showing a system enough examples for it to make predictions that generalize across the language – for example, a word following “the” in English is most likely a noun.
>
> […]
>
> POS: The simple UPOS part-of-speech tag.
>
> Tag: The detailed part-of-speech tag.

Mit dem «Part-of-speech Tagging» wird also angegeben, ob es sich bei einem Token etwa um ein Substantiv (noun), um ein Adjektiv, um ein Pronomen oder um ein Verb handelt.

`.pos_` ist dabei weniger detaillert. Da gibt es 17 Kategorien, die auf [dieser Webseite](https://universaldependencies.org/u/pos/) aufgelistet werden.

Das sind die Kategorien in `doc1`:

In [10]:
for token in doc1:
  print(f"{token.text} -> {token.pos_}")

Der -> DET
Berg -> NOUN
hat -> AUX
eine -> DET
Maus -> NOUN
geboren -> VERB
! -> PUNCT


Um eine Erklärung dieser Kategorien zu bekommen, kann man die Funktion `spacy.explain()` benutzen:

In [11]:
for token in doc1:
  print(f"{token.text} -> {token.pos_} = {spacy.explain(token.pos_)}")

Der -> DET = determiner
Berg -> NOUN = noun
hat -> AUX = auxiliary
eine -> DET = determiner
Maus -> NOUN = noun
geboren -> VERB = verb
! -> PUNCT = punctuation


(Das würde auch mit `.tag_` funktionieren, wenn uns die detaillierteren Kategorien interessieren.)

Somit lassen sich auch sprachliche Eigenschaften von Texten untersuchen: z.B., wie oft kommen Adjektive vor? Was sind die häufigsten Verben? Charaktisieren sich gewisse Textsorten (oder etwa die Reden einer bestimmten Partei) dadurch, dass sie mehr Adverben verwenden als andere?

Part-of-speech Tagging kann auch eingesetzt werden, um die Analyse auf eine Kategorie von Wörtern zu beschränken. Vielleicht wollen wir wissen, welche *Substantive* am häufigsten vorkamen. Oder vielleicht interessieren wir uns für «Fliegen», d.h. die Insekten, und wollen deshalb das Verb «fliegen» ausschliessen.

Wie gut die Tags von spaCy sind, hängt nicht nur vom Modell ab (grössere Modelle geben bessere Ergebnisse, sind aber langsamer), sondern auch von der OCR-Qualität.

#### 1.3.4 Aufgabe

Schreibe Code, der im folgenden Text `text_for_exercise` das Lemma alle Verben anzeigt. Lass Dich von den vorherigen Beispielen inspirieren!

In [12]:
text_for_exercise = """Gegenwärtig haben die Zollorgane für jedes vorübergehend
aus dem Ausland eingeführte Fahrzeug auf Grund des Bundesbeschlusses vom 20. Januar
1953 eine sogenannte Schadenbehandlungsgebühr von 3 Franken zu erheben. Mit
dieser Schadenbehandlungsgebühr hat es folgende Bewandtnis : Sie wissen, dass in
der Schweiz für Motorfahrzeuge eine Dritt-Haftpflichtversicherung vorgeschrieben
ist. Nicht alle Länder kennen die obligatorische Haftpflichtversicherung für
ihre Motorfahrzeuge. Es kommt daher vor, dass Ausländer ohne Versicherung und
vermögenslos mit ihrem Fahrzeug einen Unfall verursachen. Der Schweizer, der
dabei zu Schaden kommt, könnte also nicht entschädigt werden. Um dies zu
verhindern, wird dem ausländischen Automobilisten bei seiner Einreise in die Schweiz
Gelegenheit geboten, eine Versicherung abzuschliessen, so dass für allfällige
Schäden eine Deckung vorhanden ist."""

doc3 = nlp(text_for_exercise)

# Schreibe hier Deinen Code:



**Mögliche Lösung**

<details>
<summary>Lösung aufklappen</summary>

```python
doc3 = nlp(text_for_exercise)

for token in doc3:
  if token.pos_ == "VERB":
    print(token.lemma_)
```

### 1.4 Einen Text in Sätze zergliedern

SpaCy kann einen Text auch in Sätze zergliedern. Auch hier liefert spaCy bessere Ergebnisse als die, die wir mit einfachen Python-Funktionen erreichen könnten.

Zuerst zu Erinnerung, was `text2` enhält (hier etwas schöner angezeigt)

In [13]:
line = []
for i, word in enumerate(text2.split(" ")):
  line.append(word)
  if sum(len(w) for w in line) > 70:
    print(" ".join(line))
    line = []

print(" ".join(line))

Der Artikel 48 des Bundesgesetzes vom 1. Oktober 1925 über das Zollwesen, bzw. Artikel
111 der bezüglichen Vollziehungsverordnung sehen im Abfertigungsverfahren eine
ganze Anzahl Vereinfachungen und Erleichterungen vor. Der Artikel 14 des gleichen
Gesetzes, bzw. Artikel 11 der Vollziehungsverordnung sagt uns, was und welche Waren-Sendungen
und Warenmengen zollfrei seien.


Eine einfache Herangehensweise könnte darin bestehen, den Text anhand der Punkte zu splitten:

In [15]:
text2.split(".")

['Der Artikel 48 des Bundesgesetzes vom 1',
 ' Oktober 1925 über das Zollwesen, bzw',
 ' Artikel 111 der bezüglichen Vollziehungsverordnung sehen im Abfertigungsverfahren eine ganze Anzahl Vereinfachungen und Erleichterungen vor',
 ' Der Artikel 14 des gleichen Gesetzes, bzw',
 ' Artikel 11 der Vollziehungsverordnung sagt uns, was und welche Waren-Sendungen und Warenmengen zollfrei seien',
 '']

Wir sehen aber sofort die Grenzen dieses Verfahrens. Hier steht ein Punkt in einem Datum («1. Oktober 1925»), zwei andere Punkte stehen nach der Abkürzung «bzw.». SpaCy erkennt solche Punkte und zergliedert den Text in sprachlich relevante Sätze:

In [16]:
sentences = []

for sentence in doc2.sents:
  sentences.append(sentence.text)

sentences

['Der Artikel 48 des Bundesgesetzes vom 1. Oktober 1925 über das Zollwesen, bzw. Artikel 111 der bezüglichen Vollziehungsverordnung sehen im Abfertigungsverfahren eine ganze Anzahl Vereinfachungen und Erleichterungen vor.',
 'Der Artikel 14 des gleichen Gesetzes, bzw. Artikel 11 der Vollziehungsverordnung sagt uns, was und welche Waren-Sendungen und Warenmengen zollfrei seien.']

Ähnlich wie bei den Tokens ist jedes Element von `doc2.sents` ein Objekt, in diesem Fall `Span`-Objekte, die wiederum einige Attribute ([Liste in der Dokumentation](https://spacy.io/api/span#attributes)) haben. Anders als bei `Token` sind diese Attribute für uns nicht unbedingt extrem relevant. Interessant ist aber, dass `Span`-Objekte Zugriff auf «ihre» Tokens anbieten:

In [17]:
for sentence in doc2.sents:
  print(f"Tokens im Satz '{' '.join(sentence.text.split(' ')[:10])}…'")
  print(f"Zweites Token: '{sentence[1].text}', Lemma: '{sentence[1].lemma_}', POS: {sentence[1].pos_}")
  print(f"Achtes Token: '{sentence[7].text}', Lemma: '{sentence[7].lemma_}', POS: {sentence[7].pos_}")
  print()

Tokens im Satz 'Der Artikel 48 des Bundesgesetzes vom 1. Oktober 1925 über…'
Zweites Token: 'Artikel', Lemma: 'Artikel', POS: NOUN
Achtes Token: 'Oktober', Lemma: 'Oktober', POS: NOUN

Tokens im Satz 'Der Artikel 14 des gleichen Gesetzes, bzw. Artikel 11 der…'
Zweites Token: 'Artikel', Lemma: 'Artikel', POS: NOUN
Achtes Token: 'bzw.', Lemma: 'bzw.', POS: CCONJ



### 1.5 Named Entity Recognition

SpaCy kann auch sogenannte Entitäten in einem Text erkennen. Dazu heisst es in der [Dokumentation von spaCy](https://spacy.io/usage/linguistic-features#named-entities):

> spaCy features an extremely fast statistical entity recognition system, that assigns labels to contiguous spans of tokens. The default trained pipelines can identify a variety of named and numeric entities, including companies, locations, organizations and products.

Named Entity Recognition, kurz NER, besteht also darin, Satzteile (ein Wort oder auch mehrer Wörter zusammen) als Namen von bestimmten, eindeutig identifizierbaren Objekten, Personen oder Orten (sogenannten «Entitäten») zu erkennen. Was gesucht wird und welchen Kategorien die Entitäten zugeordnet werden, hängt vom Modell ab. Im Fall des kleinen deutschen vortrainiertes Modell von spaCy, das wir hier verwenden, gibt es vier Kategorien: `LOC` (Orte), `ORG` (Institutionen, Organisationen), `PER` (Personen) und `MISC` (sonstiges…).

Die Nutzung ist ziemlich einfach, das Attribut `.ents` funktioniert wieder wie eine Liste, und jedes Element von `.ents` ist ein `Span`-Objekt. Interessante Attribute von Entitäten-`Span`-Objekte sind etwa `.text` und `.label_`.

In [18]:
text4 = """Vaterlaus, Berichterstatter: Um den heute zur Behandlung stehenden
Bundesbeschluss über die Gewährung einer Hilfe an die Holzverzuckerungs AG zu
beurteilen, scheint mir eine kurze Zusammenfassung der Entstehungsgeschichte
und der Entwicklung der Holzverzuckerungs AG notwendig zu sein. Wie Sie der
ausführlichen Botschaft des Bundesrates haben entnehmen können, wurde auch in
der Schweiz vor dem letzten Kriege die Herstellung eines Ersatztreibstoffes
durch chemische Gewinnung von Holzzucker und dessen Umwandlung in Sprit geprüft
und versucht. Der Versuch, Alkohol dem Benzin beizumischen, löste 1935 scharfe
Kämpfe aus. Die Initianten verzichteten darum auf die Errichtung eines
eigentlichen Treibstoffwerkes. Trotzdem wurde im Winter 1935/36 an den Kleinen
Rat des Kantons Graubünden das Begehren gestellt, ein Projekt für die
Herstellung einer kleineren Anlage zur Erzeugung von jährlich 20 000 Hektoliter
Feinsprit zu fördern. Da ein solches Werk volkswirtschaftlich für den Kanton
Graubünden von grosser Bedeutung war, ersuchte der Kleine Rat am 8. September
1937 das Eidgenössische Finanz- und Zolldepartement um die Erteilung einer
Konzession und um Abnahme des anfallenden Sprits bis zu 20 000 Hektoliter
jährlich durch die Alkoholverwaltung, und zwar zu den Gestehungskosten."""

doc4 = nlp(text4.replace("\n", " "))

for entity in doc4.ents:
  print(f"{entity.text} ({entity.label_})")

Vaterlaus (LOC)
Holzverzuckerungs AG (ORG)
Holzverzuckerungs AG (ORG)
der Schweiz (LOC)
Sprit (LOC)
Kleinen Rat des Kantons Graubünden (LOC)
Feinsprit (LOC)
Kanton Graubünden (LOC)
Kleine Rat (LOC)
Eidgenössische Finanz- und Zolldepartement (ORG)


Auch hier kann man `spacy.explain()` aufrufen, um besser zu wissen, was die Kategorien "LOC" oder "ORG" bedeuten:

In [19]:
print("LOC:", spacy.explain("LOC"))
print("ORG:", spacy.explain("ORG"))

LOC: Non-GPE locations, mountain ranges, bodies of water
ORG: Companies, agencies, institutions, etc.


Hier erkennt man die Grenzen des kleinen Sprachmodells, das wir hier anwenden.

- Es wird zwar erkannt, dass «Vaterlaus» ein Eigenname ist, aber nicht, dass es sich um eine Person handelt.
- Der Kleine Rat des Kantons Graubünden wird als Ort `LOC` bezeichnet, obwohl es eigentlich `ORG` sein sollte.
- «Sprit» und «Feinsprit» werden aus irgendeinem Grund als `LOC` bezeichnet.

Die Dokumentation von spaCy sagt dazu:

> Because models are statistical and strongly depend on the examples they were trained on, this doesn’t always work perfectly and might need some tuning later, depending on your use case.

## 2 Wörter zählen

Mit dem, was wir bisher gesehen haben, ist es schon gut möglich, zu zählen, wie oft ein bestimmtes Wort in einem Text vorkommt (jedes Vorkommen wird in der Linguistik *Okkurrenz* genannt). Aber es ist noch eine etwas grössere Herausforderung, die Anzahl der Okkurrenzen jedes Wortes, das im Text vorkommt, zu zählen (also ohne im Voraus ein bestimmtes Wort anzugeben).

Um dieses Problem elegant zu lösen, muss hier ein zusätzlicher Datentyp von Python eingeführt werden: **Dictionaries**.

### 2.1 Dictionaries

Das Problem, das wir hier lösen müssen, ist Folgendes: Wie verbinde ich eine Information (in diesem Fall eine Zahl, die Anzahl von Okkurrenzen) mit einer anderen Information (das Wort als Zeichenkette)?

Eventuell liesse sich das so lösen, dass ich zwei Listen erstelle, und darauf achte, dass die Zahl immer in der gleichen Position (am gleichen Index) in der ersten Liste zu finden ist wie das Wort in der zweiten.

Python bietet aber eine viele bessere Möglichkeit: sogenannte Dictionaries. Wie ein «Wörterbuch», das zu jedem Wort eine Definition enthält, kann ein Dictionary in Python einen beliebigen Wert mit einem anderen beliebigen Wert verknüpfen.

Hier ein einfaches Beispiel:

In [20]:
zahlen_zu_text = { 1: "eis", 2: "zwöi", 3: "drü", 4: "vier" }

Ein Dictionary wird also so definiert:
- Es werden geschweifte Klammern `{}` verwendet.
- Es gibt Informationen, die als Schlüssel (*keys*) dienen. In diesem Fall sind es die Zahlen 1 bis 4. Nach jedem Schlüssel kommt ein Doppelpunkt.
- Nach dem Doppelpunkt kommt der Wert, der dem Schlüssel zugeordnet wird. In diesem Fall sind es Zeichenketten (`"eis"`, `"zwöi"`, usw.)
- Zwischen zwei Zuordnungen kommt ein Komma.

Hier ein zweites Beispiel, aus dem Kurs Python für Historiker:innen:

In [22]:
books = {"Franz Kafka": "Der Prozess",
         "Charles Dickens": "Eine Geschichte aus zwei Städten",
         "J. K. Rowling": [ "Harry Potter und der Stein der Weisen",
                          "Harry Potter und die Kammer des Schreckens",
                          "Harry Potter und der Gefangene von Askaban" ] }

Wie es dort erklärt wird:

> Die Schlüssel (<i>keys</i>) sind in diesem Beispiel die Namen der Autor:innen und die ihnen jeweils zugewiesenen Werte (<i>values</i>) sind korrespondierende Buchtitel.

Hier wird ersichtlich, dass die Werte unterschiedler Datentype sein können, und dass Verschachtelungen möglich sind: der Wert, der dem Schlüssel `"J. K. Rowling"` zugewiesen wird, ist eine Liste, während die Werte der anderen Schlüssel Zeichenketten sind.

Ähnlich wie bei Listen kann auf die einzelnen Werte dank eckigen Klammern `[]` zugegriffen werden. Anders als bei Listen wird aber keine Position angegeben, sondern der Schlüssel:



In [23]:
books["Franz Kafka"]

'Der Prozess'

In [24]:
# in diesem Fall wird eine Zahl angegeben; die Zahl ist aber nicht die Positionsangabe,
# sondern der Schlüssel, der definiert worden ist

zahlen_zu_text[3]

'drü'

Ein Dictionary kann auch verändert werden. Es kann ein Schlüssel hinzugefügt werden:

In [25]:
books["Albert Camus"] = "Die Pest"

In [26]:
books["Albert Camus"]

'Die Pest'

Der Wert kann auch verändert werden:

In [27]:
books["Albert Camus"] = ["Die Pest", "Der Fremde"]

Man kann auch überprüfen, ob ein Schlüssel im Dictionary vorhanden ist:

In [28]:
if "Franz Kafka" in books:
  print("Franz Kafka found in `books`")
else:
  print("Franz Kafka not found in `books`")

if "Wolfgang Goethe" in books:
  print("Wolfgang Goethe found in `books`")
else:
  print("Wolfgang Goethe not found in `books`")

Franz Kafka found in `books`
Wolfgang Goethe not found in `books`


**Aufgabe**: Lies im Kurs «Python für Historiker:innen», wie man sonst mit Dictionaries umgeht: Kapitel [Dictionaries](https://digital-history-berlin.github.io/Python-fuer-Historiker-innen/ch03-komplexe-datenstrukturen/05-dictionaries.html). Du kannst die nächste Codezelle verwenden, um die Python-Anweisungen aus dem Kapitel auszuprobieren:

In [ ]:
# Probiere aus, wie man mit Dictionaries arbeitet



### 2.2 `collections.Counter`

Wie in der letzten Aufgabe des Kapitels in «Python für Historiker:innen» ersichtlich wird, kann mit einem Dictionary schon eine einfache Frequenzanalyse durchgeführt werden. Das könnte etwa so aussehen:

In [29]:
# leeres Dictionary
frequencies = {}

for token in doc2:
  # wenn `token.text` noch nicht als Schlüssel angelegt ist
  # setze die Anzahl auf 1
  if token.text not in frequencies:
    frequencies[token.text] = 1
  else:
    # sonst wird weiter gezählt
    frequencies[token.text] = frequencies[token.text] + 1

frequencies

{'Der': 2,
 'Artikel': 4,
 '48': 1,
 'des': 2,
 'Bundesgesetzes': 1,
 'vom': 1,
 '1.': 1,
 'Oktober': 1,
 '1925': 1,
 'über': 1,
 'das': 1,
 'Zollwesen': 1,
 ',': 3,
 'bzw.': 2,
 '111': 1,
 'der': 2,
 'bezüglichen': 1,
 'Vollziehungsverordnung': 2,
 'sehen': 1,
 'im': 1,
 'Abfertigungsverfahren': 1,
 'eine': 1,
 'ganze': 1,
 'Anzahl': 1,
 'Vereinfachungen': 1,
 'und': 3,
 'Erleichterungen': 1,
 'vor': 1,
 '.': 2,
 '14': 1,
 'gleichen': 1,
 'Gesetzes': 1,
 '11': 1,
 'sagt': 1,
 'uns': 1,
 'was': 1,
 'welche': 1,
 'Waren-Sendungen': 1,
 'Warenmengen': 1,
 'zollfrei': 1,
 'seien': 1}

Im Modul `collections` bietet Python aber auch ein spezielles Dictionary für diesen Zweck, der ganz logischerweise `Counter` heisst.

In [30]:
from collections import Counter

Der Code kann dann vereinfacht werden:

In [31]:
# leeres Dictionary
frequencies2 = Counter()

for token in doc2:
  # Anders als vorher müssen wir nicht überprüfen, ob `token.text` im Counter
  # schon vorhanden ist. Wenn nicht, sorgt Counter selber dafür, dass die
  # Frequenz implizit 0 ist, solange kein Wert dem Schlüssel zugewiesen worden ist
  frequencies2[token.text] = frequencies2[token.text] + 1

frequencies2

Counter({'Der': 2,
         'Artikel': 4,
         '48': 1,
         'des': 2,
         'Bundesgesetzes': 1,
         'vom': 1,
         '1.': 1,
         'Oktober': 1,
         '1925': 1,
         'über': 1,
         'das': 1,
         'Zollwesen': 1,
         ',': 3,
         'bzw.': 2,
         '111': 1,
         'der': 2,
         'bezüglichen': 1,
         'Vollziehungsverordnung': 2,
         'sehen': 1,
         'im': 1,
         'Abfertigungsverfahren': 1,
         'eine': 1,
         'ganze': 1,
         'Anzahl': 1,
         'Vereinfachungen': 1,
         'und': 3,
         'Erleichterungen': 1,
         'vor': 1,
         '.': 2,
         '14': 1,
         'gleichen': 1,
         'Gesetzes': 1,
         '11': 1,
         'sagt': 1,
         'uns': 1,
         'was': 1,
         'welche': 1,
         'Waren-Sendungen': 1,
         'Warenmengen': 1,
         'zollfrei': 1,
         'seien': 1})

`Counter()` hat auch die praktische Methode `most_common()`. Schau Dir die [Dokumentation](https://docs.python.org/3/library/collections.html#collections.Counter.most_common) zu dieser Funktion an.

In [32]:
# Zeige mir die 5 Tokens, die am meisten vorkommen:
frequencies2.most_common(5)

[('Artikel', 4), (',', 3), ('und', 3), ('Der', 2), ('des', 2)]

`Counter()` kann auch eine Liste weitergegeben werden, und es wird dann automatisch gezählt:

In [33]:
all_tokens = []

for token in doc2:
  all_tokens.append(token.text)

Counter(all_tokens)

Counter({'Der': 2,
         'Artikel': 4,
         '48': 1,
         'des': 2,
         'Bundesgesetzes': 1,
         'vom': 1,
         '1.': 1,
         'Oktober': 1,
         '1925': 1,
         'über': 1,
         'das': 1,
         'Zollwesen': 1,
         ',': 3,
         'bzw.': 2,
         '111': 1,
         'der': 2,
         'bezüglichen': 1,
         'Vollziehungsverordnung': 2,
         'sehen': 1,
         'im': 1,
         'Abfertigungsverfahren': 1,
         'eine': 1,
         'ganze': 1,
         'Anzahl': 1,
         'Vereinfachungen': 1,
         'und': 3,
         'Erleichterungen': 1,
         'vor': 1,
         '.': 2,
         '14': 1,
         'gleichen': 1,
         'Gesetzes': 1,
         '11': 1,
         'sagt': 1,
         'uns': 1,
         'was': 1,
         'welche': 1,
         'Waren-Sendungen': 1,
         'Warenmengen': 1,
         'zollfrei': 1,
         'seien': 1})

### 2.3 Aufgabe

Verändere folgenden Code, um anstatt alle mögliche Tokens zu zählen, **nur die Tokens zu zählen, die keine Satzzeichen und keine Stoppwörter sind**.

In [34]:
text_for_final_exercise = """Ferner hat der Bundesrat, um den Automobilisten der Pflicht zur Sicherstellung der Zollabgabe beim Grenzübertritt zu entheben,
am 24. August 1926 ein Reglement erlassen, nach welchem Motorfahrzeuge aller Art auf Grund von internationalen und nationalen Zollpapieren vorübergehend
zur Einfuhr abgefertigt werden, für welche Touristenverbände oder Versicherungsgesellschaften der Zollverwaltung für geschuldete Abgaben Sicherheit leisten.
Im weitern erheben die Zollverwaltungen seit dem 1. Juni 1949 im Sinne einer provisorischen Massnahme zur Beschleunigung der Abfertigung im Reisenden- und
Touristenverkehr alle auf der Einfuhr lastenden Abgaben in Form von Pauschalansätzen. In diesen Pauschalansätzen sind auch die Warenumsatzsteuer und die
Stempelgebühren Inbegriffen. Zudem besteht seit dem 1. Januar 1950, auf Grund einer Entschliessung der Organisation für europäische wirtschaftliche
Zusammenarbeit in Paris, für die Einfuhr bestimmter Mengen von Wein, Spirituosen und Tabakwaren Abgabefreiheit. Ferner können Reiseandenken bis zum
Gesamtwert von 400 Franken von europäischen Touristen bzw. 2000 Franken von Überseetouristen ohne esondere Zollabfertigung aus der Schweiz ausgeführt oder
durch unser Land im Transit mitgenommen werden. Diese Erleichterungen beziehen sich aber nur auf ausländische Touristen, die zu einem vorübergehenden
Aufenthalt in die Schweiz einreisen, nicht aber auf die in der Schweiz wohnhaften, von einem Auslandsaufenthalt zurückkehrenden Personen. Das alles sind
Erleichterungen, die die bisherige Zollgesetzgebung vorgesehen hat. Und nun hat sich seit dem Kriegsende der internationale Reise- und Touristenverkehr
ganz gewaltig vermehrt. Zudem setzt jedes Jahr mit Beginn der Feriensaison ein enormer Verkehr ein, der zu unliebsamen Störungen und Verkehrsstockungen
führt. Wir müssen daher feststellen, dass die bisher gewährten Erleichterungen den heutigen Verhältnissen in keiner Weise mehr Rechnung tragen. Es ist
deshalb verständlich, wenn in letzter Zeit von vielen Seiten der Ruf nach gewissen Lockerungen der Zollkontrolle immer lauter geworden ist. Auch der
Zollverwaltung sind diese Lockerungen erwünscht, weil sie gezwungen ist, während der Reisesaison die Strassenzollämter mit vermehrtem Grenzwachtpersonal
zu verstärken und damit die Grenzbewachung zu schwächen. Zudem ist das Ende dieser ausserordentlichen Verkehrsentwicklung nicht abzusehen. Im
Gegenteil, wir müssen in den nächsten Jahren zweifellos mit einer weiteren starken Zunahme rechnen. Dies trifft vor allem im Motorfahrzeugverkehr zu."""

doc5 = nlp(text_for_final_exercise.replace("\n", " "))

words = []

for token in doc5:
  words.append(token.text)

frequencies5 = Counter(words)

for word, count in frequencies5.most_common(10):
  print(f"'{word}' kommt {count} Mal vor.")

',' kommt 17 Mal vor.
'der' kommt 16 Mal vor.
'.' kommt 15 Mal vor.
'die' kommt 12 Mal vor.
'von' kommt 8 Mal vor.
'und' kommt 7 Mal vor.
'in' kommt 7 Mal vor.
'zu' kommt 6 Mal vor.
'auf' kommt 5 Mal vor.
' ' kommt 4 Mal vor.


Hier ist das erhoffte Ergebnis:

```
'Einfuhr' kommt 3 Mal vor.
'Zudem' kommt 3 Mal vor.
'Schweiz' kommt 3 Mal vor.
'Erleichterungen' kommt 3 Mal vor.
'Ferner' kommt 2 Mal vor.
'Grund' kommt 2 Mal vor.
'Zollverwaltung' kommt 2 Mal vor.
'Abgaben' kommt 2 Mal vor.
'1.' kommt 2 Mal vor.
'Touristenverkehr' kommt 2 Mal vor.
```

Was merkst Du? Was würdest Du vielleicht noch gern anpassen oder verändern? Lass Dich auch von der Vorstellung von spaCy inspirieren (Kapitel 1 in diesem Notebook).